In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
 
from sklearn.linear_model    import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics         import (mean_absolute_error,
                                     mean_squared_error, r2_score)
import random
random.seed(42)
np.random.seed(42)

In [2]:
#  1. GÉNÉRATION DU DATASET — Prix de Maisons
 
n = 200
 
surface    = np.random.randint(30, 200, n)
nb_pieces  = np.clip(surface // 20 + np.random.randint(-1, 2, n), 1, 10)
age        = np.random.randint(0, 50, n)
etage      = np.random.randint(0, 10, n)
quartier   = np.random.choice(['Centre', 'Nord', 'Sud', 'Est', 'Ouest'], n)
 
# Prix réaliste : surface est le facteur dominant
bruit  = np.random.normal(0, 15000, n)
prix   = (surface * 3500
          + nb_pieces * 8000
          - age * 500
          + etage * 2000
          + bruit).astype(int)
prix   = np.clip(prix, 80000, 800000)
 
df = pd.DataFrame({
    'surface':   surface,
    'nb_pieces': nb_pieces,
    'age':       age,
    'etage':     etage,
    'quartier':  quartier,
    'prix':      prix,
})
 
print(f"Dataset : {df.shape[0]} maisons × {df.shape[1]} colonnes")
print(df.describe().round(0))

Dataset : 200 maisons × 6 colonnes
       surface  nb_pieces    age  etage      prix
count    200.0      200.0  200.0  200.0     200.0
mean     113.0        5.0   25.0    4.0  435576.0
std       48.0        2.0   14.0    3.0  191015.0
min       30.0        1.0    0.0    0.0   91230.0
25%       73.0        3.0   13.0    2.0  274074.0
50%      116.0        5.0   25.0    4.0  431671.0
75%      158.0        7.0   36.0    7.0  602980.0
max      199.0       10.0   49.0    9.0  778309.0


In [3]:
#  2. PRÉPARATION DES DONNÉES
 
# Features numériques uniquement pour ce premier modèle
X = df[['surface', 'nb_pieces', 'age', 'etage']]
y = df['prix']
 
# Séparation train / test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
 
print(f"\n Train : {len(X_train)} maisons")
print(f" Test  : {len(X_test)} maisons")


 Train : 160 maisons
 Test  : 40 maisons


In [4]:
#  3. ENTRAÎNEMENT DU MODÈLE
 
model = LinearRegression()
model.fit(X_train, y_train)
 
print("\n Coefficients du modèle :")
for feature, coef in zip(X.columns, model.coef_):
    print(f"   {feature:<12} : {coef:>10.0f} €")
print(f"   {'intercept':<12} : {model.intercept_:>10.0f} €")


 Coefficients du modèle :
   surface      :       3521 €
   nb_pieces    :       8581 €
   age          :       -513 €
   etage        :       1602 €
   intercept    :      -2216 €


In [5]:
#  4. ÉVALUATION DU MODÈLE
 
y_pred = model.predict(X_test)
 
r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
 
print(f"\n Performances sur le jeu de test :")
print(f"   R²   = {r2:.3f}  (1.0 = parfait)")
print(f"   MAE  = {mae:,.0f} €  (erreur moyenne absolue)")
print(f"   RMSE = {rmse:,.0f} €  (erreur quadratique moyenne)")
 
# Prédiction exemple
exemple = pd.DataFrame([[90, 4, 10, 2]], columns=X.columns)
pred    = model.predict(exemple)[0]
print(f"\n Prédiction — 90m², 4 pièces, 10 ans, étage 2 :")
print(f"   Prix estimé : {pred:,.0f} €")


 Performances sur le jeu de test :
   R²   = 0.993  (1.0 = parfait)
   MAE  = 13,568 €  (erreur moyenne absolue)
   RMSE = 16,715 €  (erreur quadratique moyenne)

 Prédiction — 90m², 4 pièces, 10 ans, étage 2 :
   Prix estimé : 347,043 €


In [6]:
#  5. VISUALISATION — 4 GRAPHIQUES
 
BG     = '#0F172A'
CARD   = '#1E293B'
VERT   = '#00FF94'
BLEU   = '#00E0FF'
VIOLET = '#A78BFA'
ROUGE  = '#FF6B6B'
TEXTE  = '#E2E8F0'
GRIS   = '#94A3B8'
 
def style(ax):
    ax.set_facecolor(CARD)
    ax.tick_params(colors=GRIS, labelsize=9)
    ax.xaxis.label.set_color(GRIS)
    ax.yaxis.label.set_color(GRIS)
    for s in ['top', 'right']:  ax.spines[s].set_visible(False)
    for s in ['bottom', 'left']: ax.spines[s].set_color('#334155')
 
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor(BG)
fig.suptitle('Régression Linéaire — Prix Maisons · Jour 7/30',
             fontsize=14, fontweight='bold', color=TEXTE, y=0.98)
 
#G1 : Surface vs Prix (scatter + droite)
ax1 = axes[0, 0]; style(ax1)
ax1.scatter(df['surface'], df['prix'],
            color=VIOLET, alpha=0.4, s=30, label='Données')
# Droite de régression surface → prix
m  = LinearRegression().fit(df[['surface']], df['prix'])
xs = np.linspace(df['surface'].min(), df['surface'].max(), 100)
ax1.plot(xs, m.predict(xs.reshape(-1, 1)),
         color=VERT, linewidth=2.5, label='Régression')
ax1.set_xlabel('Surface (m²)');  ax1.set_ylabel('Prix (€)')
ax1.set_title('Surface vs Prix', color=TEXTE, fontweight='bold')
ax1.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda v, _: f'{v/1000:.0f}k€'))
ax1.legend(facecolor=CARD, labelcolor=TEXTE, fontsize=9)
 
#G2 : Réel vs Prédit
ax2 = axes[0, 1]; style(ax2)
ax2.scatter(y_test, y_pred, color=BLEU, alpha=0.6, s=40)
lim = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax2.plot(lim, lim, color=VERT, linewidth=2, linestyle='--',
         label='Prédiction parfaite')
ax2.set_xlabel('Prix réel (€)'); ax2.set_ylabel('Prix prédit (€)')
ax2.set_title(f'Réel vs Prédit  (R²={r2:.2f})',
              color=TEXTE, fontweight='bold')
ax2.xaxis.set_major_formatter(
    plt.FuncFormatter(lambda v, _: f'{v/1000:.0f}k€'))
ax2.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda v, _: f'{v/1000:.0f}k€'))
ax2.legend(facecolor=CARD, labelcolor=TEXTE, fontsize=9)
 
# G3 : Résidus
ax3 = axes[1, 0]; style(ax3)
residus = y_test - y_pred
ORANGE = '#F97316'
ax3.scatter(y_pred, residus, color=ORANGE,
            alpha=0.6, s=40)
ax3.axhline(y=0, color=VERT, linewidth=2, linestyle='--')
ax3.set_xlabel('Valeurs prédites (€)')
ax3.set_ylabel('Résidus (€)')
ax3.set_title('Analyse des Résidus', color=TEXTE, fontweight='bold')
ax3.xaxis.set_major_formatter(
    plt.FuncFormatter(lambda v, _: f'{v/1000:.0f}k€'))
ax3.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda v, _: f'{v/1000:.0f}k€'))
 
# G4 : Importance des feature
ax4 = axes[1, 1]; style(ax4)
features  = list(X.columns)
coefs_abs = np.abs(model.coef_)
colors4   = [VERT, BLEU, VIOLET, ROUGE]
bars = ax4.barh(features, coefs_abs,
                color=colors4, edgecolor=BG, height=0.5)
ax4.set_title('Importance des Variables (|coef|)',
              color=TEXTE, fontweight='bold')
ax4.set_xlabel('Valeur absolue du coefficient', color=GRIS)
for bar in bars:
    w = bar.get_width()
    ax4.text(w + 20, bar.get_y() + bar.get_height() / 2,
             f'{w:,.0f}', va='center', color=GRIS, fontsize=9)
 
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('dashboard_regression.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
print('\n Dashboard exporté → dashboard_regression.png')
plt.show()

C:\Users\maath\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(



 Dashboard exporté → dashboard_regression.png


C:\Users\maath\AppData\Local\Temp\ipykernel_5548\4185357284.py:89: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
